# 1D kCSD (adapted from `CSD to share to cursor.ipynb`)

Same kCSD maths as your working notebook:
- per unique `x` column → `KCSD1D`
- `y` normalized to \[0, 1\]
- temporal Gaussian smooth (`sigma=1`)
- `h=0.1`, `n_src=2*n_ch`
- interpolate CSD back onto electrode depths

**Kernel:** `Python (kCSD_env)` — check `sys.executable` points to `...\envs\kCSD_env\python.exe`.

For delta peak/trough data from MATLAB Estimate, only set `DELTA_ROOT` below.


In [ ]:
import sys
print("executable:", sys.executable)
if "kCSD_env" not in sys.executable.replace("\\", "/"):
    print("WARNING: this kernel does not look like kCSD_env — change kernel before continuing.")


In [ ]:
# Only if imports fail in kCSD_env (run once in a terminal, not every session):
#   conda activate kCSD_env
#   python -m pip install -U -r requirements.txt


In [ ]:

# ---------------------------------------------------------
# 0) IMPORTS  (same packages as the share notebook + pipeline helper)
# ---------------------------------------------------------
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from csd_1d_pipeline import (
    KCSD1D,
    DELTA_CSD_DEFAULTS,
    SHARE_NOTEBOOK_KCSD,
    discover_delta_csd_datasets,
    run_delta_csd_from_root,
    run_dataset,
)

# Fail early with a clear message if kcsd is missing in this env
assert KCSD1D is not None, (
    "KCSD1D could not be imported. Activate kCSD_env / select that kernel, "
    "then: python -m pip install -U -r requirements.txt"
)
print("KCSD1D OK:", KCSD1D)
print("Share-notebook kCSD settings:", SHARE_NOTEBOOK_KCSD)


## A) Delta peak/trough root (MATLAB Estimate output)

Only change this path. Uses the share-notebook kCSD settings with a
**−500 … +500 ms** window and **baseline correction** (pre-T0 mean).
No re-reference.


In [ ]:
# ---------------------------------------------------------
# 1) USER SETTINGS — delta export folder
# ---------------------------------------------------------
# Window / baseline are set in csd_1d_pipeline.DELTA_CSD_DEFAULTS:
#   t_pre = t_post = 500  →  -500 … +500 ms around T0
#   baseline_correct = True
# (Ignore DatasetConfig's 200/800 — those are only for stim/Lombard.)
DELTA_ROOT = r"\\experimentfs.home.bccn-berlin.de\experiment\PlayNeuralData\NPX-OPTO PLAY NMM\PlayBout Analysis\play bout analysis behavior and physiology\Data\Analysis results\delta peak trough CSD"

datasets = discover_delta_csd_datasets(DELTA_ROOT)
print(f"Found {len(datasets)} peak/trough folders")
print("DELTA_CSD_DEFAULTS window:", DELTA_CSD_DEFAULTS["t_pre"], DELTA_CSD_DEFAULTS["t_post"],
      "baseline", DELTA_CSD_DEFAULTS["baseline_correct"])
for d in datasets[:12]:
    print(" ", d["name"])
if len(datasets) > 12:
    print(" ...")


In [ ]:
# Window: -500 .. +500 ms around T0; baseline_correct=True (DELTA_CSD_DEFAULTS)

# ---------------------------------------------------------
# 2–5) LOAD → EPOCH → 1D kCSD per column → PLOT/SAVE
#      (same algorithm as cells 1–3 of the share notebook)
# ---------------------------------------------------------
results = run_delta_csd_from_root(
    DELTA_ROOT,
    plot=True,          # show example column like the share notebook
    stop_on_error=False,
)

ok = [r for r in results if "error" not in r]
bad = [r for r in results if "error" in r]
print(f"Finished: {len(ok)} ok, {len(bad)} failed")
for r in bad:
    print(" -", r["name"], ":", r["error"])


## B) Single Lombard/stim folder (exact share-notebook style)

Use this to re-run one of your old folders the same way as
`CSD to share to cursor.ipynb` (with reference + baseline).


In [ ]:

# ---------------------------------------------------------
# Optional: one stim folder, mimicking the share notebook settings
# ---------------------------------------------------------
# main_folder = r"//experimentfs.home.bccn-berlin.de/experiment/share/Miguel/LOMBARD EFFECT NPX files and analysis/CSD analysis/230516/USV CSD/"
# cfg = {
#     **SHARE_NOTEBOOK_KCSD,
#     "folder": main_folder,
#     "name": "230516_USV",
#     "t_pre": 200,
#     "t_post": 800,
#     "ref_chan": 0,
#     "channels_remove": (191,),
#     "chan_start": 0,
#     "chan_end": 383,
#     "baseline_correct": True,
# }
# single = run_dataset(cfg)


## C) Behavior/play-bout onset and offset (self-locating)

Run this section after MATLAB finishes `Estimate_behavior_aligned_csd`.

It locates the repository code even when the Jupyter kernel starts in another folder. Timing is **−1 s to +2 s** around each onset and offset; both alignments use the same **−1 s to 0 s onset baseline**. Input LFP is raw and unfiltered.

In [ ]:
# Self-locating imports: this cell does not assume the kernel's working folder.
import sys
from pathlib import Path

NETWORK_REPO_ROOTS = [
    Path(r"\\experimentfs.home.bccn-berlin.de\experiment\PlayNeuralData\NPX-OPTO PLAY NMM\PlayBout Analysis\play bout analysis behavior and physiology"),
    Path(r"\\experimentfs.bccn-berlin.pri\experiment\PlayNeuralData\NPX-OPTO PLAY NMM\PlayBout Analysis\play bout analysis behavior and physiology"),
]

code_candidates = []
for start in [Path.cwd(), *NETWORK_REPO_ROOTS]:
    code_candidates.extend([start, start / "CSD analysis"])
    code_candidates.extend(parent / "CSD analysis" for parent in start.parents)

CODE_DIR = next(
    (path for path in code_candidates if (path / "behavior_aligned_kcsd.py").is_file()),
    None,
)
if CODE_DIR is None:
    raise FileNotFoundError(
        "Could not locate CSD analysis/behavior_aligned_kcsd.py. "
        "Update NETWORK_REPO_ROOTS in this cell if the repository moved."
    )

sys.path.insert(0, str(CODE_DIR))

# Reload so rerunning this section picks up pipeline updates
# without depending on stale imports from an earlier notebook execution.
import importlib
import csd_1d_pipeline as _pipeline
import behavior_aligned_kcsd as _behavior_pipeline
import area_csd_consistency as _area_csd
importlib.reload(_pipeline)
importlib.reload(_behavior_pipeline)
importlib.reload(_area_csd)

KCSD1D = _pipeline.KCSD1D
discover_behavior_aligned_pairs = _behavior_pipeline.discover_behavior_aligned_pairs
run_behavior_aligned_kcsd_from_root = _behavior_pipeline.run_behavior_aligned_kcsd_from_root
average_saved_behavior_kcsd_from_root = _behavior_pipeline.average_saved_behavior_kcsd_from_root
organize_behavior_aligned_figures = _behavior_pipeline.organize_behavior_aligned_figures
summarize_area_csd_across_sessions = _area_csd.summarize_area_csd_across_sessions
summarize_session_area_overlap = _area_csd.summarize_session_area_overlap
finalize_area_consistency_from_cache = _area_csd.finalize_area_consistency_from_cache
organize_area_consistency_figures = _area_csd.organize_area_consistency_figures

assert KCSD1D is not None, (
    "KCSD1D could not be imported. Select the kCSD_env kernel and install "
    "CSD analysis/requirements.txt."
)
print("Code folder:", CODE_DIR)
print("Python:", sys.executable)
print("Behavior/kCSD modules reloaded")

In [ ]:
# Data folder. Set this to a string only if automatic discovery fails.
BEHAVIOR_ROOT_OVERRIDE = None

if BEHAVIOR_ROOT_OVERRIDE:
    BEHAVIOR_ROOT = Path(BEHAVIOR_ROOT_OVERRIDE)
else:
    data_candidates = [
        root / "Data" / "Analysis results" / "behavior aligned CSD"
        for root in NETWORK_REPO_ROOTS
    ]
    BEHAVIOR_ROOT = next((path for path in data_candidates if path.is_dir()), None)

if BEHAVIOR_ROOT is None or not BEHAVIOR_ROOT.is_dir():
    raise FileNotFoundError(
        "Behavior-aligned MATLAB output was not found. First run "
        "Estimate_behavior_aligned_csd in MATLAB, or set "
        "BEHAVIOR_ROOT_OVERRIDE above."
    )

# Explicit here as well as in the module defaults, so notebook behavior is clear.
T_PRE_MS = 1000.0
T_POST_MS = 2000.0
ONSET_BASELINE_MS = (-1000.0, 0.0)
SAVE_PLOTS = True

print("Behavior data root:", BEHAVIOR_ROOT)
print("Epoch: -1000 to +2000 ms; shared onset baseline: -1000 to 0 ms")

In [ ]:
pairs = discover_behavior_aligned_pairs(BEHAVIOR_ROOT)
if not pairs:
    raise FileNotFoundError(
        f"No paired onset/offset lfp.npy exports found below {BEHAVIOR_ROOT}"
    )

print(f"Found {len(pairs)} paired datasets")
for pair in pairs[:20]:
    print(" ", pair["name"])
if len(pairs) > 20:
    print(f"  ... and {len(pairs) - 20} more")

In [ ]:
results = run_behavior_aligned_kcsd_from_root(
    BEHAVIOR_ROOT,
    stop_on_error=False,
    t_pre=T_PRE_MS,
    t_post=T_POST_MS,
    onset_baseline_window_ms=ONSET_BASELINE_MS,
    ref_chan=None,
    baseline_correct=True,
    normalize="none",
    compute_kcsd=True,
    compute_fd=False,
    save=True,
    save_plots=SAVE_PLOTS,
    save_column_plots=False,  # keep matrices, but only plot the averaged probe view
    save_mean_plot=True,
    plot=False,
    stop_on_missing_event_ids=True,
    retain_epochs=False,
    event_chunk_size=8,
)

ok = [result for result in results if "error" not in result]
failed = [result for result in results if "error" in result]
print(f"Finished: {len(ok)} successful, {len(failed)} failed")
for result in failed:
    print(" -", result["name"], ":", result["error"])

if failed:
    print("Successful datasets were still saved; inspect the failures above.")

### Average existing matrices and sort PNG figures — no kCSD re-estimation

If the per-column kCSD matrices already exist, **skip the long estimation cell above** and run the next cell. It:

1. loads the existing `csd_x*.npy` matrices;
2. averages staggered columns without fitting kCSD again;
3. removes individual-column PNGs and writes only `kcsd_xmean.png`;
4. rebuilds the two copied exploration views:
   - `sorted_figures/by_session/onset|offset/`
   - `sorted_figures/by_behavior/onset|offset/`;
5. writes `sorted_figures/figure_index.csv` linking each averaged source PNG to both copies.

In [ ]:
# This cell does NOT call KCSD1D and does not re-estimate CSD.
averaging_results = average_saved_behavior_kcsd_from_root(
    BEHAVIOR_ROOT,
    output_subdir="csd_1d_onset_baseline",
    save_plots=True,
    remove_column_plots=True,
    stop_on_error=False,
)

figure_sorting = organize_behavior_aligned_figures(
    BEHAVIOR_ROOT,
    output_folder_name="sorted_figures",
    csd_output_subdir="csd_1d_onset_baseline",
    refresh=True,
)

failed_averaging = [r for r in averaging_results if "error" in r]
print(f"Averaging failures: {len(failed_averaging)}")
figure_sorting

### Sampled areas and overlap between recordings

Run this before the area-consistency cell. It does not fit kCSD and it does not modify `Area_limits_GoodLooking.xlsx`. For each recording it lists the anatomical areas reached by the saved electrode depths. Probes from the same recording are combined.

The overlap matrix is asymmetric. Entry `a(i, j)` is the percentage of areas sampled by recording `i` that are also sampled by recording `j`.

A pair is kept only when both directions are at least 50%. One direction is not enough: a short recording can lie completely inside a longer one while the longer one shares only a small part of its own areas. The selected group is the largest set in which every pair passes that test. If two sets have the same size, the higher mean overlap is kept, and a remaining tie uses the alphabetically first session list. Recordings outside that group are not averaged together in the next cell.

In [ ]:
# Reads y_mean_columns.npy and the area table. It does not fit kCSD or write the workbook.
area_table_candidates = [
    CODE_DIR.parent / "Data" / "Area_limits_GoodLooking.xlsx",
    *(
        root / "Data" / "Area_limits_GoodLooking.xlsx"
        for root in NETWORK_REPO_ROOTS
    ),
]
AREA_TABLE = next((path for path in area_table_candidates if path.is_file()), None)
if AREA_TABLE is None:
    raise FileNotFoundError(
        "Area_limits_GoodLooking.xlsx was not found. Looked in:\n"
        + "\n".join(str(path) for path in area_table_candidates)
    )

session_overlap = summarize_session_area_overlap(
    BEHAVIOR_ROOT,
    AREA_TABLE,
    output_subdir="csd_1d_onset_baseline",
    min_shared_area_percent=50,
)
print(session_overlap["reason"])
print("Selected sessions:", session_overlap["selected_sessions"])
print("Overlap matrix:", session_overlap["overlap_csv"])
session_overlap["overlap_figure"]

### Area consistency across sessions — no kCSD re-estimation

Run this after the overlap cell and after `csd_mean_columns.npy` exists. It reads those saved maps, assigns depths with `Data/Area_limits_GoodLooking.xlsx` using the same animal-key and depth rules as the other MATLAB analyses, averages probes within a session, then compares only the recordings selected above. The spreadsheet is only read. This cell never saves or overwrites `Area_limits_GoodLooking.xlsx`.

A probe segment is used only when its unique depths span at least 200 µm. Each kept segment is also smoothed with a Gaussian of 250 ms by 100 µm full width at half maximum. Where at least five sessions overlap, a one-sample t-test marks bins different from zero. Clusters are kept when they are at least 25 ms long and cover at least 10% of the area's relative depth. Positive and negative bins are clustered separately. The outline is drawn on `smoothed_mean_clusters.png`. The unsmoothed mean and sign-consistency figures remain.

The figures keep the saved kCSD sign (`RdBu_r`). They do not relabel red/blue as source or sink. Sign-consistency images are blank wherever fewer than two sessions overlap. Onset and offset of the same area share one color scale.

Each accepted session map is saved under `area_consistency/session_cache/` as it is read. If Windows cannot delete an old `area_consistency` folder on experimentFS, this cell overwrites the files instead of stopping. The run that already failed on that delete did not write a cache. Rerun this cell once. After the cache exists, the next cell rebuilds the means, clusters, and copies without reading `csd_mean_columns.npy` again.

In [ ]:
# This cell does NOT call KCSD1D and does not re-estimate CSD.
# It needs csd_mean_columns.npy. If that file is missing, run the averaging cell above first.
# Area_limits_GoodLooking.xlsx is read into memory and is never saved or overwritten.
area_table_candidates = [
    CODE_DIR.parent / "Data" / "Area_limits_GoodLooking.xlsx",
    *(
        root / "Data" / "Area_limits_GoodLooking.xlsx"
        for root in NETWORK_REPO_ROOTS
    ),
]
AREA_TABLE = next((path for path in area_table_candidates if path.is_file()), None)
if AREA_TABLE is None:
    raise FileNotFoundError(
        "Area_limits_GoodLooking.xlsx was not found. Looked in:\n"
        + "\n".join(str(path) for path in area_table_candidates)
    )

area_consistency = summarize_area_csd_across_sessions(
    BEHAVIOR_ROOT,
    AREA_TABLE,
    output_subdir="csd_1d_onset_baseline",
    output_folder_name="area_consistency",
    reuse_session_cache=True,
    session_allowlist=session_overlap["selected_sessions"],
)
print("Area table:", AREA_TABLE)
print("Area maps:", area_consistency["n_groups"])
print("Summary:", area_consistency["summary_csv"])
area_consistency["summary_csv"]

### Resume area means and figures from the session cache

Run this when `area_consistency/session_cache/` already exists and you only need the across-session means, cluster outlines, and `by_behavior` copies. It does not read `csd_mean_columns.npy`, does not open the area spreadsheet, and does not fit kCSD.

The `PermissionError` on experimentFS happened while deleting `area_consistency`, after the maps had been read into memory and before anything was written. That memory is gone, and the old code did not save a cache, so this cell cannot recover that attempt. Rerun the area cell above once. If a later step stops while writing figures, rerun this cell instead of reading every saved map again.

In [ ]:
# Rebuilds area means, clusters, and by_behavior copies from session_cache.
# It does not read csd_mean_columns.npy, the area table, or fit kCSD.
# Only recordings chosen by the session-overlap cell are pooled.
selected_path = Path(BEHAVIOR_ROOT) / "session_overlap" / "selected_sessions.txt"
if not selected_path.is_file():
    raise FileNotFoundError(
        "No selected_sessions.txt. Run the session-overlap cell before resuming."
    )
selected_sessions = [
    line.strip() for line in selected_path.read_text(encoding="utf-8").splitlines() if line.strip()
]
area_consistency = finalize_area_consistency_from_cache(
    BEHAVIOR_ROOT,
    output_folder_name="area_consistency",
    session_allowlist=selected_sessions,
)
print("Area maps:", area_consistency["n_groups"])
print("Summary:", area_consistency["summary_csv"])
print("Sorted figures:", area_consistency["sorted_figures"]["output_root"])
area_consistency["summary_csv"]

### Browse merged area maps by behavior

The area cell above keeps one folder per scope, behavior, alignment, and area. This cell copies the merged PNGs into two folders so onset and offset can be scrolled without opening each area:

- `area_consistency/by_behavior/onset/`
- `area_consistency/by_behavior/offset/`

Each filename starts with the behavior: `behavior__scope__area__mean_csd.png`. The smoothed cluster image is copied the same way, with `__smoothed_clusters` at the end of the name. The original area folders stay where they are. Run this cell alone when those maps already exist.

In [ ]:
# Copies existing merged area PNGs. It does not estimate kCSD or reread the area table.
area_figure_sorting = organize_area_consistency_figures(
    BEHAVIOR_ROOT,
    area_folder_name="area_consistency",
    output_folder_name="by_behavior",
    refresh=True,
)
area_figure_sorting